In [3]:
# ---
# title: COCO to YOLOv8, YOLOv9, YOLOv10 Converter
# description: Converts COCO JSON annotations to YOLO text format and JSON metadata

# ---

import json
import os
from tqdm import tqdm

# === CONFIGURATION ===
coco_json_path = "final_training_sets_coco/instances_val.json"  # path to COCO JSON file
images_dir = "final_training_sets_coco/valid_images"                           # directory containing the images
output_dir = "final_training_sets_yolo/valid"                      # where YOLO txt files will be saved
output_metadata_json = True                           # whether to save YOLOv8/9/10-style dataset JSON
yolo_version = "v8"                                   # or "v9", "v10" (all same format here)

os.makedirs(output_dir, exist_ok=True)

# === LOAD COCO DATA ===
with open(coco_json_path, "r") as f:
    coco_data = json.load(f)

categories = {cat["id"]: cat["name"] for cat in coco_data["categories"]}
images = {img["id"]: img for img in coco_data["images"]}

# Create label index mapping
category_to_idx = {cat_id: idx for idx, cat_id in enumerate(categories.keys())}

print(f"Loaded {len(images)} images and {len(coco_data['annotations'])} annotations.")


# === CONVERT FUNCTION ===
def coco_bbox_to_yolo(bbox, img_w, img_h):
    """
    Convert COCO bbox [x_min, y_min, width, height]
    to YOLO format [x_center, y_center, width, height] normalized [0,1].
    """
    x, y, w, h = bbox
    x_c = x + w / 2
    y_c = y + h / 2
    return [x_c / img_w, y_c / img_h, w / img_w, h / img_h]


# === GROUP ANNOTATIONS BY IMAGE ===
annotations_by_image = {}
for ann in coco_data["annotations"]:
    img_id = ann["image_id"]
    if img_id not in annotations_by_image:
        annotations_by_image[img_id] = []
    annotations_by_image[img_id].append(ann)

# === CONVERT TO YOLO ===
for img_id, anns in tqdm(annotations_by_image.items(), desc="Converting"):
    img_info = images[img_id]
    img_w, img_h = img_info["width"], img_info["height"]
    img_filename = os.path.splitext(img_info["file_name"])[0]

    yolo_lines = []
    for ann in anns:
        cat_id = ann["category_id"]
        yolo_cat = category_to_idx[cat_id]
        bbox_yolo = coco_bbox_to_yolo(ann["bbox"], img_w, img_h)
        yolo_line = f"{yolo_cat} " + " ".join(f"{x:.6f}" for x in bbox_yolo)
        yolo_lines.append(yolo_line)

    # Save YOLO txt
    txt_path = os.path.join(output_dir, f"{img_filename}.txt")
    with open(txt_path, "w") as f:
        f.write("\n".join(yolo_lines))

print(f" Conversion completed! Labels saved in: {output_dir}")


# === OPTIONAL: CREATE YOLOv8/9/10 JSON METADATA ===
if output_metadata_json:
    yolo_json = {
        "train": images_dir,
        "val": images_dir.replace("train", "val"),
        "nc": len(categories),
        "names": list(categories.values())
    }

    json_path = os.path.join(os.path.dirname(output_dir), f"dataset_{yolo_version}.json")
    with open(json_path, "w") as f:
        json.dump(yolo_json, f, indent=4)

    print(f" YOLO {yolo_version} metadata JSON saved at: {json_path}")


Loaded 12 images and 18 annotations.


Converting: 100%|████████████████████████████████████████████████████████████████████| 12/12 [00:00<00:00, 1415.24it/s]

 Conversion completed! Labels saved in: final_training_sets_yolo/valid
 YOLO v8 metadata JSON saved at: final_training_sets_yolo\dataset_v8.json
